# 01 — Raw dataset profile (Task A2)

Profiles `data/raw/universal_top_spotify_songs.csv` (Asaniczka, *Top Spotify Songs in 73 Countries — Daily*) and reconciles what we find against the assumptions in `SPEC.md §7`.

Runs top-to-bottom against the raw file. Uses **synthetic-free** real data; the file is gitignored, so this notebook only runs after Task A1 places the CSV.

Reports, per the A2 acceptance criteria: column list, dtypes, row count, null fraction per column, cardinality of artist/album/country, date-column min/max, fraction of rows with a valid ISRC, fraction with audio features — then a one-paragraph verdict on whether the dataset matches `SPEC.md §7`.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

# Walk up to the repo root (the dir that contains `src/`) so `from src import config` works
# regardless of whether the notebook is launched from repo root or from notebooks/.
ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config

RAW_PATH = config.RAW_DIR / config.PRIMARY_DATASET_FILENAME
print("repo root :", ROOT)
print("raw file  :", RAW_PATH)
print("exists    :", RAW_PATH.is_file())
print("pandas    :", pd.__version__)

repo root : D:\Projects\spotify-trend-analysis-dashboard
raw file  : D:\Projects\spotify-trend-analysis-dashboard\data\raw\universal_top_spotify_songs.csv
exists    : True
pandas    : 3.0.3


## 1. Load

In [2]:
# ~498 MB; low_memory=False avoids mixed-dtype chunk warnings on wide string columns.
df = pd.read_csv(RAW_PATH, low_memory=False)
print(f"rows x cols: {df.shape[0]:,} x {df.shape[1]}")
print(f"memory     : {df.memory_usage(deep=True).sum() / 1e6:,.0f} MB")

rows x cols: 2,110,316 x 25


memory     : 1,224 MB


## 2. Columns + dtypes

In [3]:
schema = pd.DataFrame({"dtype": df.dtypes.astype(str)})
schema.index.name = "column"
schema

,dtype
column,
spotify_id,str
name,str
artists,str
daily_rank,int64
daily_movement,int64
weekly_movement,int64
country,str
snapshot_date,str
popularity,int64


## 3. Null fraction per column

In [4]:
null_frac = df.isna().mean().sort_values(ascending=False)
null_report = (null_frac * 100).round(3).to_frame("null_%")
null_report

,null_%
country,1.370
album_name,0.039
album_release_date,0.031
name,0.001
artists,0.001
daily_movement,0.000
weekly_movement,0.000
spotify_id,0.000
daily_rank,0.000
popularity,0.000


## 4. Cardinality of key categoricals

In [5]:
card_cols = [c for c in ["spotify_id", "name", "artists", "album_name", "country"] if c in df.columns]
cardinality = {c: df[c].nunique(dropna=True) for c in card_cols}
pd.Series(cardinality, name="distinct_values").to_frame()

,distinct_values
spotify_id,24983
name,21809
artists,13726
album_name,16440
country,72


In [6]:
# Multi-artist tracks: the `artists` field is a delimited list. Estimate how many rows have >1 artist.
if "artists" in df.columns:
    multi = df["artists"].fillna("").str.contains(",").mean()
    print(f"rows whose `artists` contains a comma (multi-artist): {multi * 100:.1f}%")
    print("\nsample `artists` values:")
    print(df["artists"].dropna().drop_duplicates().head(8).to_list())

rows whose `artists` contains a comma (multi-artist): 40.7%

sample `artists` values:


['Alex Warren', 'Sabrina Carpenter', 'sombr', 'Lady Gaga, Bruno Mars', 'Billie Eilish', 'Jin', 'W Sound, Beéle, Ovy On The Drums', 'ROSÉ, Bruno Mars']


## 5. Date columns — min/max

In [7]:
for col in ["snapshot_date", "album_release_date"]:
    if col in df.columns:
        parsed = pd.to_datetime(df[col], errors="coerce", format="mixed")
        unparsed = parsed.isna().sum() - df[col].isna().sum()
        print(f"{col:20s} min={parsed.min()}  max={parsed.max()}  unparseable(non-null)={max(unparsed, 0)}")

snapshot_date        min=2023-10-18 00:00:00  max=2025-06-11 00:00:00  unparseable(non-null)=0


album_release_date   min=1900-01-01 00:00:00  max=2025-07-18 00:00:00  unparseable(non-null)=0


## 6. ISRC coverage

`SPEC.md §7` makes ISRC the **primary** track key. Check whether the column even exists, and whether `spotify_id` can serve as the de-facto stable key instead.

In [8]:
if "isrc" in df.columns:
    valid_isrc = df["isrc"].astype(str).str.fullmatch(r"[A-Za-z]{2}[A-Za-z0-9]{3}\d{7}").mean()
    print(f"valid ISRC fraction: {valid_isrc * 100:.1f}%")
else:
    print("!! No `isrc` column in this dataset.")
    if "spotify_id" in df.columns:
        sid = df["spotify_id"]
        print(f"   spotify_id non-null : {sid.notna().mean() * 100:.2f}%")
        print(f"   spotify_id distinct : {sid.nunique():,}")
        # A spotify track id is 22 base62 chars.
        looks_like_id = sid.astype(str).str.fullmatch(r"[A-Za-z0-9]{22}").mean()
        print(f"   matches 22-char base62 id pattern: {looks_like_id * 100:.2f}%")

!! No `isrc` column in this dataset.
   spotify_id non-null : 100.00%


   spotify_id distinct : 24,983


   matches 22-char base62 id pattern: 100.00%


## 7. Audio-feature coverage

In [9]:
present_feats = [c for c in config.AUDIO_FEATURE_COLS if c in df.columns]
missing_feats = [c for c in config.AUDIO_FEATURE_COLS if c not in df.columns]
print("config.AUDIO_FEATURE_COLS present:", present_feats)
print("config.AUDIO_FEATURE_COLS missing:", missing_feats or "none")

if present_feats:
    feat_nonnull = df[present_feats].notna().mean().mul(100).round(3).to_frame("non_null_%")
    display(feat_nonnull)
    all_present = df[present_feats].notna().all(axis=1).mean()
    print(f"\nrows with ALL audio features present: {all_present * 100:.2f}%")

extra_audio = [c for c in ["key", "mode", "time_signature"] if c in df.columns]
print("extra audio columns in data but not in config:", extra_audio or "none")

config.AUDIO_FEATURE_COLS present: ['danceability', 'energy', 'valence', 'tempo', 'acousticness', 'liveness', 'speechiness', 'instrumentalness', 'loudness']
config.AUDIO_FEATURE_COLS missing: none


,non_null_%
danceability,100.0
energy,100.0
valence,100.0
tempo,100.0
acousticness,100.0
liveness,100.0
speechiness,100.0
instrumentalness,100.0
loudness,100.0



rows with ALL audio features present: 100.00%
extra audio columns in data but not in config: ['key', 'mode', 'time_signature']


## 8. Country / geography shape

In [10]:
if "country" in df.columns:
    blank = df["country"].isna() | (df["country"].astype(str).str.strip() == "")
    print(f"rows with blank country (the 'Global' chart): {blank.mean() * 100:.1f}%")
    print(f"distinct non-blank country codes           : {df.loc[~blank, 'country'].nunique()}")
    print("\nsample country codes:", sorted(df.loc[~blank, "country"].dropna().unique())[:15])

rows with blank country (the 'Global' chart): 1.4%
distinct non-blank country codes           : 72



sample country codes: ['AE', 'AR', 'AT', 'AU', 'BE', 'BG', 'BO', 'BR', 'BY', 'CA', 'CH', 'CL', 'CO', 'CR', 'CZ']


## 9. Fact-measure sanity: popularity / rank / streams

In [11]:
for col in ["popularity", "daily_rank", "daily_streams"]:
    if col in df.columns:
        s = pd.to_numeric(df[col], errors="coerce")
        print(f"{col:16s} min={s.min()} max={s.max()} null%={df[col].isna().mean() * 100:.2f}")
    else:
        print(f"{col:16s} -- ABSENT from dataset")

popularity       min=0 max=100 null%=0.00
daily_rank       min=1 max=50 null%=0.00
daily_streams    -- ABSENT from dataset


## 10. Reconciliation against `SPEC.md §7`

In [12]:
# Canonical name (used downstream by clean/transform) -> raw column in THIS dataset (or None if absent).
EXPECTED = {
    "track_name": "name",
    "primary_artist_name": "artists",  # delimited list; clean.py extracts the primary
    "isrc": None,
    "spotify_track_id": "spotify_id",
    "explicit": "is_explicit",
    "duration_ms": "duration_ms",
    "album_name": "album_name",
    "release_date": "album_release_date",
    "snapshot_date": "snapshot_date",
    "country": "country",
    "popularity": "popularity",
    "rank": "daily_rank",
    "daily_streams": None,
    "primary_genre": None,
    **{f: f for f in config.AUDIO_FEATURE_COLS},
}
recon = pd.DataFrame(
    [
        {
            "canonical": k,
            "raw_column": v,
            "present": (v in df.columns) if v is not None else False,
        }
        for k, v in EXPECTED.items()
    ]
)
recon

,canonical,raw_column,present
0,track_name,name,True
1,primary_artist_name,artists,True
2,isrc,NaN,False
3,spotify_track_id,spotify_id,True
4,explicit,is_explicit,True
5,duration_ms,duration_ms,True
6,album_name,album_name,True
7,release_date,album_release_date,True
8,snapshot_date,snapshot_date,True
9,country,country,True


In [13]:
absent = recon.loc[~recon["present"], "canonical"].to_list()
extra = sorted(set(df.columns) - set(v for v in EXPECTED.values() if v is not None))
summary = (
    f"PROFILE SUMMARY -- does the dataset match SPEC.md §7?\n"
    f"  rows={df.shape[0]:,}, cols={df.shape[1]}, snapshot max={pd.to_datetime(df['snapshot_date'], errors='coerce').max().date()}\n"
    f"  PARTIAL MATCH. Audio features (the analytical thesis) are all present, plus key/mode/time_signature.\n"
    f"  But the schema diverges from SPEC §7 in ways that need a Gate-G1 decision:\n"
    f"    - assumed-but-ABSENT canonical fields: {absent}\n"
    f"    - most columns are named differently (name->track_name, artists->primary_artist_name, is_explicit->explicit,\n"
    f"      album_release_date->release_date, daily_rank->rank) -> handled by a raw->canonical rename map in clean.py.\n"
    f"    - NO ISRC: SPEC's ISRC-first key strategy is infeasible; `spotify_id` is the natural stable track key.\n"
    f"    - NO daily_streams and NO genre: 'size by streams' and 'box plot by genre' charts lose an encoding.\n"
    f"    - `country` is blank for the Global chart; per-country rows carry ISO-2 codes.\n"
    f"  extra raw columns not in the canonical model: {extra}"
)
print(summary)

PROFILE SUMMARY -- does the dataset match SPEC.md §7?
  rows=2,110,316, cols=25, snapshot max=2025-06-11
  PARTIAL MATCH. Audio features (the analytical thesis) are all present, plus key/mode/time_signature.
  But the schema diverges from SPEC §7 in ways that need a Gate-G1 decision:
    - assumed-but-ABSENT canonical fields: ['isrc', 'daily_streams', 'primary_genre']
    - most columns are named differently (name->track_name, artists->primary_artist_name, is_explicit->explicit,
      album_release_date->release_date, daily_rank->rank) -> handled by a raw->canonical rename map in clean.py.
    - NO ISRC: SPEC's ISRC-first key strategy is infeasible; `spotify_id` is the natural stable track key.
    - NO daily_streams and NO genre: 'size by streams' and 'box plot by genre' charts lose an encoding.
    - `country` is blank for the Global chart; per-country rows carry ISO-2 codes.
  extra raw columns not in the canonical model: ['daily_movement', 'key', 'mode', 'time_signature', 'weekly_m

## Verdict

**Partial match — three structural surprises for the Gate-G1 review.** The snapshot holds **2,110,316** daily chart rows spanning **2023-10-18 → 2025-06-11**, covering **24,983** distinct tracks across **72** countries plus a blank-country *Global* chart. The analytical core is intact: all nine features in `AUDIO_FEATURE_COLS` are present and **100% non-null**, and the data adds three more (`key`, `mode`, `time_signature`). `popularity` is a clean 0–100 with no nulls; `daily_rank` is a top-50 chart position.

Three assumptions in `SPEC.md §7` do **not** hold:

1. **No ISRC column.** The ISRC-first key strategy is infeasible. `spotify_id` (100% present, 100% valid 22-char base62) is the natural stable track key; dedup falls back to (`track_name` + primary artist) only if an id were ever missing. *This touches SPEC's "Always use ISRC" boundary and needs team sign-off.*
2. **No `daily_streams` and no genre.** The Mood Map's *size = streams* encoding and the *box plot by genre* chart (SPEC §8) each lose a channel — substitute *size = popularity* and bucket by release-year instead of genre.
3. **Most columns are renamed** (`name`→`track_name`, `artists`→`primary_artist_name`, `is_explicit`→`explicit`, `album_release_date`→`release_date`, `daily_rank`→`rank`). `artists` is a comma-delimited list (**40.7%** of rows are multi-artist), so `clean.py` must extract the primary artist.

Reconciled in `src/config.py` via a raw→canonical rename map (Task A3) and recorded in `powerbi/data_model.md`; the contentious items above are flagged for Gate G1.